# 06 · Explainable AI: which input matters, and how

Both methods use the **held-out** outer test folds of the main runs and the models saved in notebook 03 (each fold's model explains only its own test boreholes).

| Method | What it measures |
|---|---|
| Permutation importance | drop in balanced accuracy when one input is shuffled; also for the groups **location** (Easting + Northing together) and **elevation** |
| SHAP (exact Shapley values) | how much each input raises or lowers P(Clay) for each cell; the background is a sample of the fold's training cells |
| Elevation dependence | SHAP of elevation against elevation, next to the observed clay share: a geotechnical plausibility check |

**Run time:** 20-60 minutes (SHAP of the GW models is the slow part). Set `XAI_MODELS` to fewer models to make it faster.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import joblib
import shap
from sklearn.metrics import balanced_accuracy_score

XAI_MODELS = list(S["cv"]["models"])     # e.g. ["GWXGB", "XGB"] for a faster run
N_REPEATS = 10                           # shuffles per input and fold
SHAP_CELLS_PER_FOLD = S["quick"]["shap_cells"] if S["run"]["quick"] else 200
BACKGROUND = 100                         # training cells used as the SHAP background

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
folds, block_m = H.load_folds(S, 1)
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
preds = preds[(preds.multiple == 1) & (preds.purge_x == 0)]
MODELS = H.out_dir(S, "03_spatial_cv", private=True) / "models"
OUT, PRIV = H.out_dir(S, "06_xai"), H.out_dir(S, "06_xai", private=True)
SHORT = {"UTM_Easting_m": "Easting", "UTM_Northing_m": "Northing", "Top_Elevation_m": "Elevation"}
GROUPS = {"Easting": ["UTM_Easting_m"], "Northing": ["UTM_Northing_m"], "Elevation": ["Top_Elevation_m"],
          "Location (E+N)": ["UTM_Easting_m", "UTM_Northing_m"]}

## 1. Permutation importance (held-out)

In [ ]:
rng = np.random.default_rng(S["run"]["seed"])
rows = []
for name in XAI_MODELS:
    for k, test in preds[preds.model == name].groupby("outer_fold"):
        model = joblib.load(MODELS / f"{name}_x1_purge0_fold{k}.joblib")
        thr = test.threshold.iloc[0]
        for subset, t in (("all", test), ("measured", test[test.measured == 1])):
            base = balanced_accuracy_score(t.y, (model.p_clay(t) >= thr).astype(int))
            for label, cols in GROUPS.items():
                drops = []
                for _ in range(N_REPEATS):
                    shuffled = t.copy()
                    order = rng.permutation(len(t))
                    shuffled[cols] = t[cols].values[order]          # one permutation for the whole group
                    drops.append(base - balanced_accuracy_score(t.y, (model.p_clay(shuffled) >= thr).astype(int)))
                rows.append({"model": name, "outer_fold": k, "cells": subset, "input": label,
                             "kind": "group" if len(cols) > 1 else "single", "drop": float(np.mean(drops))})
    print(name, "done", flush=True)
perm = pd.DataFrame(rows)
perm_sum = perm.groupby(["model", "cells", "kind", "input"]).drop.agg(["mean", "std"]).reset_index()
perm_sum.to_csv(OUT / "permutation_importance.csv", index=False)
display(perm_sum[perm_sum.cells == "all"].pivot_table(index="model", columns="input", values="mean").round(3))

## 2. SHAP values (held-out cells)

In [ ]:
shap_rows = []
for name in XAI_MODELS:
    for k, test in preds[preds.model == name].groupby("outer_fold"):
        model = joblib.load(MODELS / f"{name}_x1_purge0_fold{k}.joblib")
        train, _ = H.outer_split(cells, folds, k)
        bg = train.sample(min(BACKGROUND, len(train)), random_state=k)[H.FEATURES].values
        ex = test.sample(min(SHAP_CELLS_PER_FOLD, len(test)), random_state=k)
        f = lambda X: model.p_clay(pd.DataFrame(X, columns=H.FEATURES))
        sv = shap.explainers.Exact(f, shap.maskers.Independent(bg, max_samples=BACKGROUND))(ex[H.FEATURES].values, silent=True)
        out = ex[["BH_Code", "cell", "depth_top_m", "measured", "y", *H.FEATURES]].copy()
        for j, c in enumerate(H.FEATURES):
            out[f"shap_{SHORT[c]}"] = sv.values[:, j]
        shap_rows.append(out.assign(model=name, outer_fold=k, base=sv.base_values))
    print(name, "done", flush=True)
shap_df = pd.concat(shap_rows, ignore_index=True)
shap_df.to_csv(PRIV / "shap_values_by_cell.csv", index=False)
mean_abs = shap_df.groupby("model")[[f"shap_{v}" for v in SHORT.values()]].apply(lambda d: d.abs().mean())
mean_abs.to_csv(OUT / "shap_mean_abs.csv")
display(mean_abs.round(4))

## 3. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mean_abs.plot.bar(ax=axes[0], color=["#17365C", "#17918C", "#A0004E"])
axes[0].set_title("Mean |SHAP| on P(Clay)"); axes[0].set_xlabel("")
p = perm_sum[(perm_sum.cells == "all")].pivot_table(index="model", columns="input", values="mean")
p[["Location (E+N)", "Elevation"]].plot.bar(ax=axes[1], color=["#17918C", "#A0004E"])
axes[1].set_title("Permutation importance: drop in balanced accuracy"); axes[1].set_xlabel("")
fig.tight_layout(); fig.savefig(OUT / "fig_importance.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(1, len(XAI_MODELS) + 1, figsize=(3.2 * (len(XAI_MODELS) + 1), 4), sharey=True)
bins = pd.cut(cells.Top_Elevation_m, 20)
prof = cells.groupby(bins, observed=True).agg(z=("Top_Elevation_m", "mean"), clay=("y", "mean"))
axes[0].plot(prof.clay, prof.z, "-o", color="black", markersize=3)
axes[0].set_xlabel("Observed clay share"); axes[0].set_ylabel("Top elevation of cell (m)")
for ax, name in zip(axes[1:], XAI_MODELS):
    d = shap_df[shap_df.model == name]
    ax.scatter(d.shap_Elevation, d.Top_Elevation_m, s=4, alpha=0.4, color="#A0004E")
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.set_xlabel("SHAP of elevation"); ax.set_title(name)
fig.suptitle("Elevation: observed clay share vs the models' elevation effect")
fig.tight_layout(); fig.savefig(OUT / "fig_elevation_dependence.png", dpi=200, bbox_inches="tight")
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 07.")